# Tabular — Statistik Inferensial + Uji A/B (bahan "ketepatan metode" dan presentasi)
**Kapan dipakai:** menjawab pertanyaan "apakah X benar-benar berpengaruh?" dengan p-value dan
selang kepercayaan (confidence interval, CI).

**Aturan baca:** p < 0.05 → perbedaan/hubungan **signifikan** (kecil kemungkinan hanya
kebetulan). Ingat: signifikan ≠ besar. Selalu laporkan juga ukuran efeknya.

**Data demo:** Titanic (`survived`) dan Hillstrom email marketing (uji A/B).

In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest

## Konfigurasi

In [2]:
DATA_FOLDER = "../../data_latihan/tabular/classification"   # GANTI saat lomba
DATA_CSV = f"{DATA_FOLDER}/titanic_train.csv"
AB_FOLDER = "../../data_latihan/tabular_extra/uplift"
AB_CSV = f"{AB_FOLDER}/hillstrom.csv"

TARGET = "survived"
N_BOOTSTRAP = 2000
SEED = 42

## Baca data

In [3]:
data = pd.read_csv(DATA_CSV)
print(data.shape)
data.head()

(712, 16)


,id,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,693,1,3,male,NaN,0,0,56.4958,S,Third,man,True,NaN,Southampton,yes,True
1,482,0,2,male,NaN,0,0,0.0000,S,Second,man,True,NaN,Southampton,no,True
2,528,0,1,male,NaN,0,0,221.7792,S,First,man,True,C,Southampton,no,True
3,856,1,3,female,18.0,0,1,9.3500,S,Third,woman,False,NaN,Southampton,yes,False
4,802,1,2,female,31.0,1,1,26.2500,S,Second,woman,False,NaN,Southampton,yes,False


## 1. Kategori vs kategori: uji chi-square + Cramér's V
Cramér's V = ukuran efek 0–1 (0 = tidak berhubungan, 1 = berhubungan sempurna).

In [4]:
table = pd.crosstab(data["sex"], data[TARGET])
chi2, p_chi2, degrees_of_freedom, expected = stats.chi2_contingency(table)
n_rows = table.to_numpy().sum()
cramers_v = np.sqrt(chi2 / (n_rows * (min(table.shape) - 1)))
print(table)
print("chi2 =", round(chi2, 2), "| p =", f"{p_chi2:.2e}", "| Cramér's V =", round(cramers_v, 3))

survived    0    1
sex               
female     65  188
male      374   85
chi2 = 212.38 | p = 4.16e-48 | Cramér's V = 0.546


## 2. Angka antar 2 kelompok: Welch t-test + Mann-Whitney
Mann-Whitney tidak butuh data berdistribusi normal. Cohen's d = selisih rata-rata dalam
satuan simpangan baku (0.2 kecil, 0.5 sedang, 0.8 besar).

In [5]:
survived_fare = data.loc[data[TARGET] == 1, "fare"]
died_fare = data.loc[data[TARGET] == 0, "fare"]
t_stat, p_ttest = stats.ttest_ind(survived_fare, died_fare, equal_var=False)
u_stat, p_mannwhitney = stats.mannwhitneyu(survived_fare, died_fare)
pooled_std = np.sqrt((survived_fare.var() + died_fare.var()) / 2)
cohens_d = (survived_fare.mean() - died_fare.mean()) / pooled_std
print("rata-rata fare selamat:", round(survived_fare.mean(), 1),
      "| tidak:", round(died_fare.mean(), 1))
print("Welch t p =", f"{p_ttest:.2e}", "| Mann-Whitney p =", f"{p_mannwhitney:.2e}",
      "| Cohen's d =", round(cohens_d, 2))

rata-rata fare selamat: 48.6 | tidak: 21.4
Welch t p = 1.57e-10 | Mann-Whitney p = 6.89e-20 | Cohen's d = 0.54


## 3. Angka antar >2 kelompok: ANOVA + Kruskal-Wallis
Contoh: apakah umur berbeda antar kelas tiket (pclass 1, 2, 3)?

In [6]:
groups = []
for pclass_value, group in data.groupby("pclass"):
    groups.append(group["age"].dropna())

f_stat, p_anova = stats.f_oneway(*groups)
h_stat, p_kruskal = stats.kruskal(*groups)
mean_age = data.groupby("pclass")["age"].mean()
print(mean_age.round(1))
print("ANOVA p =", f"{p_anova:.2e}", "| Kruskal p =", f"{p_kruskal:.2e}")

pclass
1    38.0
2    30.1
3    25.5
Name: age, dtype: float64
ANOVA p = 2.11e-17 | Kruskal p = 9.97e-17


## 4. Korelasi Pearson (linear) dan Spearman (urutan) + p-value

In [7]:
pairs = data[["age", "fare"]].dropna()
pearson_r, p_pearson = stats.pearsonr(pairs["age"], pairs["fare"])
spearman_rho, p_spearman = stats.spearmanr(pairs["age"], pairs["fare"])
print("Pearson r =", round(pearson_r, 3), "(p =", round(p_pearson, 3), ")")
print("Spearman rho =", round(spearman_rho, 3), "(p =", round(p_spearman, 3), ")")

Pearson r = 0.107 (p = 0.01 )
Spearman rho = 0.125 (p = 0.003 )


## 5. Regresi logistik: odds ratio tiap faktor, dengan faktor lain dikontrol
OR > 1 = menaikkan peluang selamat, OR < 1 = menurunkan. `C(...)` = kolom kategori.

In [8]:
logit = smf.logit("survived ~ C(sex) + age + C(pclass) + fare", data=data).fit(disp=0)
interval = logit.conf_int()

odds_table = pd.DataFrame()
odds_table["odds_ratio"] = np.exp(logit.params)
odds_table["ci_bawah"] = np.exp(interval[0])
odds_table["ci_atas"] = np.exp(interval[1])
odds_table["p_value"] = logit.pvalues
odds_table.to_csv("odds_ratio.csv")
odds_table.round(4)

,odds_ratio,ci_bawah,ci_atas,p_value
Intercept,48.3210,17.1220,136.3693,0.0000
C(sex)[T.male],0.0817,0.0517,0.1291,0.0000
C(pclass)[T.2],0.2487,0.1242,0.4982,0.0001
C(pclass)[T.3],0.0686,0.0328,0.1435,0.0000
age,0.9613,0.9451,0.9777,0.0000
fare,1.0007,0.9952,1.0062,0.8086


## 6. Bootstrap: selang kepercayaan 95% untuk selisih peluang selamat (wanita − pria)
Ambil sampel ulang (dengan pengembalian) ribuan kali, hitung selisihnya tiap kali.
CI 95% = persentil 2.5% sampai 97.5% dari semua selisih itu.

In [9]:
random_generator = np.random.default_rng(SEED)
women = data.loc[data["sex"] == "female", TARGET].to_numpy()
men = data.loc[data["sex"] == "male", TARGET].to_numpy()

differences = []
for i in range(N_BOOTSTRAP):
    women_sample = random_generator.choice(women, len(women))
    men_sample = random_generator.choice(men, len(men))
    differences.append(women_sample.mean() - men_sample.mean())

ci_95 = np.percentile(differences, [2.5, 97.5]).round(3)
print("selisih =", round(women.mean() - men.mean(), 3), "| CI 95% =", ci_95)

selisih = 0.558 | CI 95% = [0.492 0.621]


## 7. Banyak uji sekaligus → koreksi p-value (Holm)
Makin banyak uji, makin besar peluang "signifikan" karena kebetulan. Koreksi menaikkan p.

In [10]:
test_names = ["sex", "fare", "age_pclass", "age_fare"]
p_values = [p_chi2, p_ttest, p_anova, p_pearson]
is_significant, p_corrected, sidak_alpha, bonferroni_alpha = multipletests(p_values,
                                                                           method="holm")
correction = pd.DataFrame(index=test_names)
correction["p"] = p_values
correction["p_holm"] = p_corrected
correction["signifikan"] = is_significant
correction

,p,p_holm,signifikan
sex,4.159916e-48,1.663967e-47,True
fare,1.573756e-10,3.147512e-10,True
age_pclass,2.109661e-17,6.328984e-17,True
age_fare,1.030241e-02,1.030241e-02,True


## 8. Uji A/B: apakah email menaikkan kunjungan? (z-test dua proporsi)
Lift = kenaikan relatif tingkat kunjungan grup email dibanding kontrol.

In [11]:
ab = pd.read_csv(AB_CSV)
ab["grup"] = np.where(ab["segment"] == "No E-Mail", "kontrol", "email")
ab_summary = ab.groupby("grup")["visit"].agg(["sum", "count", "mean"])
z_stat, p_ab = proportions_ztest(ab_summary["sum"].to_numpy(), ab_summary["count"].to_numpy())
lift = ab_summary.loc["email", "mean"] / ab_summary.loc["kontrol", "mean"] - 1
print(ab_summary)
print("z =", round(z_stat, 2), "| p =", f"{p_ab:.2e}", "| lift =", f"{lift:.1%}")

          sum  count      mean
grup                          
email    7132  42694  0.167049
kontrol  2262  21306  0.106167
z = 20.51 | p = 1.75e-93 | lift = 57.3%


## 9. Ukuran sampel: berapa orang per grup agar kenaikan 10.6% → 12% terdeteksi?
alpha 0.05 = batas p. power 0.8 = 80% peluang menemukan efek kalau efeknya memang ada.

In [12]:
effect_size = proportion_effectsize(0.12, 0.106)
needed = NormalIndPower().solve_power(effect_size=effect_size, alpha=0.05, power=0.8)
print("butuh ±", int(np.ceil(needed)), "orang per grup")

butuh ± 8021 orang per grup
